# 🕸️ Notebook Guide — Structural ML & Graph Neural Networks

## Learning Objectives
- [ ] Build a protein graph representation (nodes = residues, edges = contacts)
- [ ] Implement a Message Passing Neural Network (MPNN) layer from scratch
- [ ] Build a full ProteinGNN with global pooling for property prediction
- [ ] Understand protein embedding similarity with ESM2-style embeddings
- [ ] Know the production structural biology ML stack (PyTorch Geometric, AlphaFold2, ESMFold)
- [ ] Articulate AlphaFold2's Evoformer and IPA in technical interviews

## Why GNNs for Structural Biology?
Proteins are fundamentally **graphs**: residues are nodes, contacts are edges. Standard ML (CNN, MLP) treats the input as a grid or vector — ignoring the 3D geometric relationships. GNNs:
- Naturally encode variable-length proteins (no padding needed)
- Capture local structural neighborhoods
- Are equivariant to rotations/translations (SE(3)-equivariant GNNs)
- Power AlphaFold2 (via graph-based structure module), GearNet, ESM-IF

---

## 🤖 Claude Integration — Copy These Prompts

**For Protein Graph Construction:**
```
Explain how to represent a protein as a graph for GNN processing.
For a protein with N residues:
- Nodes: what features should each residue node have?
  (amino acid identity, secondary structure, phi/psi angles, relative position)
- Edges: which residues should be connected?
  (Cα-Cα distance < 8Å, k-nearest neighbors, sequence neighbors)
- Edge features: what should they encode?
  (distance, direction vector, relative orientation)
Show me a protein graph construction class in Python.
```

**For Message Passing (core GNN operation):**
```
Explain message passing in graph neural networks.
The 3-step process:
1. Message: for each edge (i,j), compute message m_ij = f(h_i, h_j, e_ij)
2. Aggregate: gather all messages to node i: m_i = Σ_j m_ij
3. Update: new node embedding h_i' = g(h_i, m_i)
What is scatter_add and why is it used for aggregation?
Show me a MessagePassingLayer class in PyTorch.
What other aggregation functions exist? (mean, max, attention)
```

**For Global Pooling:**
```
In a GNN for protein property prediction, I have embeddings for each residue (N × d).
I need to get a single embedding for the whole protein.
What are the global pooling options?
- Global mean pooling: average all residue embeddings
- Global max pooling: take max over all residues
- Attention pooling: weighted average, learned weights
- CLS token (like Transformers): add a virtual node connected to all
When would I use each? What are the tradeoffs?
```

**For SE(3)-Equivariant GNNs:**
```
Explain equivariance in GNNs for structural biology.
What does SE(3) equivariance mean?
Why is it important for protein structure: if I rotate the protein,
the predicted function shouldn't change (invariance).
What are examples of equivariant GNNs? (EGNN, SchNet, DimeNet, SE3-Transformer, PaiNN)
How does EGNN achieve equivariance with just additional coordinate updates?
I don't need a full implementation, just a conceptual explanation.
```

**For AlphaFold2 Deep Dive:**
```
Explain AlphaFold2's architecture for a technical interview at a structural biology company.
Focus on:
1. Input representations: MSA (Multiple Sequence Alignment) + pair features
2. Evoformer: how row-attention and column-attention work on the MSA
3. The pair representation: what does it encode? (inter-residue distances/angles)
4. Structure module: Invariant Point Attention (IPA) + frame updates
5. Why does recycling (3 iterations) matter?
6. lDDT-Cα training objective: what is lDDT and why use it instead of RMSD?
Also: how does ESMFold differ from AF2? (single-sequence input, faster, no MSA needed)
```

**For Graph Attention Networks (GAT):**
```
Explain Graph Attention Networks (GAT) vs standard GNN.
In message passing, how does attention change the aggregation step?
m_i = Σ_j α_ij * message(h_j, e_ij)
where α_ij are attention weights computed from h_i and h_j.
Why is attention better than simple sum aggregation for protein graphs?
What's multi-head attention in GAT?
Show me how to add attention to the MessagePassingLayer.
```

---

## GNN Architecture Reference

```
Protein Coordinates (N atoms × 3)
        │ build_graph(threshold=8Å)
        ▼
ProteinGraph (node_feats: N×d_node, edge_feats: E×d_edge, edge_index: 2×E)
        │
        ▼
Embedding Layer (d_node → d_hidden)
        │
        ├── MessagePassingLayer × n_layers
        │   ├── msg_mlp(h_i, h_j, e_ij) → message m_ij
        │   ├── scatter_add(messages, target_nodes) → aggregated
        │   └── update_mlp(h_i, aggregated) → h_i'
        │
        ├── Global Mean Pool (N×d_hidden → d_hidden)
        │
        └── Head MLP (d_hidden → output)
```

## Production Stack for Structural Biology ML

| Library | Purpose | Key Function |
|---------|---------|-------------|
| PyTorch Geometric (PyG) | GNNs on graphs | MessagePassing base class |
| Graphein | Protein graph construction | graphein.protein.construct_graph |
| Biopython | PDB parsing | Bio.PDB.PDBParser |
| MDAnalysis | MD trajectory analysis | Universe, AtomGroup |
| AlphaFold2 | Structure prediction | evoformer, IPA |
| ESMFold / ESM2 | Single-seq structure + embeddings | esm.pretrained.esm2_t33 |
| OpenFold | Open-source AF2 | Training + inference |

---

## Interview Tip Bank
> **GNN vs Transformer for proteins**: Transformers attend to ALL pairs (O(N²)). GNNs only attend to connected neighbors (O(N×k) for k-NN graph). GNNs are better for very long proteins; Transformers for capturing long-range interactions.

> **Equivariance vs Invariance**: If rotating the input rotates the output the same way → equivariant. If rotating the input doesn't change the output → invariant. Property prediction (energy, function) should be invariant. Force prediction should be equivariant.

> **Evoformer key insight**: The MSA attention (column attention = share information across homologs) + pair attention (row attention = capture residue correlations). The two representations update each other iteratively.

> **Message passing depth**: In a k-layer GNN, each node aggregates information from its k-hop neighborhood. Too few layers = can't capture global structure. Too many = over-smoothing (all node embeddings become similar).

---

## Self-Check
1. Why is `scatter_add` needed instead of a simple sum? (variable number of neighbors per node)
2. What is the output shape of a global mean pooling layer applied to (N, d) node embeddings? (d)
3. Why is SE(3)-equivariance important for force field prediction but less important for function classification?
4. In AlphaFold2's Evoformer, what does column attention compute? (shared information across MSA rows = evolutionarily related sequences)
5. Name 3 differences between ESMFold and AlphaFold2.


## TL;DR — Plain English

**What this notebook does:** Teaches you to represent protein 3D structures as graphs and train neural networks that understand geometry — the foundation of modern protein ML.

**After this notebook you can:**
- Convert a protein PDB structure into a graph where nodes are residues and edges are contacts
- Implement a message-passing neural network (MPNN) that aggregates neighbourhood information
- Explain E(3)-equivariance: why a rotation of the protein should not change the prediction
- Train a GNN to predict protein properties from structure

**Why this matters:**
- HackerRank: Graph neural network questions (message passing, node/edge features, pooling) are increasingly tested in ML interviews
- computational biology ML teams: The Pairformer in AlphaFold3 is architecturally related to MPNNs; GNN literacy is an explicit requirement for the ML Scientist role

**Time:** ~4 hours | **Difficulty:** ⭐⭐⭐⭐ | **Prerequisites:** 05/01 (deep learning), 03/01 (structure analysis)

## Structural ML & GNNs — Concepts for Beginners

| Term | Plain English |
|------|---------------|
| **graph** | A set of nodes (things) connected by edges (relationships) — proteins modelled as residue graphs |
| **node** | A single entity in a graph (e.g. one amino acid residue) |
| **edge** | A connection between two nodes (e.g. residues within 8A of each other) |
| **node feature** | A vector of numbers describing one node (e.g. amino acid one-hot + physicochemical properties) |
| **edge feature** | A vector describing one edge (e.g. distance, angle between residues) |
| **message passing** | GNN core operation: each node collects information from its neighbours and updates its representation |
| **aggregation** | How messages from neighbours are combined — sum, mean, or max |
| **GCN** | Graph Convolutional Network — simplest GNN; averages neighbour features with learnable weights |
| **equivariance** | A model is SE(3)-equivariant if rotating the input rotates the output by the same amount |
| **invariance** | Output doesn't change when input is rotated/translated — what you want for scalar predictions (e.g. energy) |
| **k-NN graph** | Graph where each node is connected to its k nearest neighbours in 3D space |
| **contact threshold** | Distance cutoff (e.g. 8A between Calpha atoms) used to define which residues are "in contact" |

## Beginner Teaching Frame

**Who should fully work through this notebook:** students comfortable with deep learning who want to reason about relational and geometric structure.

**How to study it on a first pass:**
- understand graph construction before architecture variants
- keep invariance/equivariance intuitive before making it formal
- connect node/edge features to real protein structure meaning

**Common traps:** thinking a graph is just an adjacency matrix, skipping feature design, and treating geometric ML as a bag of acronyms.


## Canonical Resource Upgrade

Useful anchors:
- [Stanford CS224W](https://web.stanford.edu/class/cs224w/)
- [Graph Representation Learning](https://www.cs.mcgill.ca/~wlh/grl_book/)
- [PyTorch Geometric docs](https://pytorch-geometric.readthedocs.io/)


## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | Module 00/06 — PyTorch fundamentals, Module 03/01 — protein structural biology (coordinates, contact maps), Module 05/01 — deep learning & fine-tuning |
| 📍 You Are Here | Module 06/01 — Structural ML & Graph Neural Networks (MPNN, SE(3) equivariance, AlphaFold2 IPA, protein GNNs) |
| ➡️ Next Steps | Module 07/01 — AlphaFold3 core concepts, Module 10/01 — OpenFold3 fine-tuning capstone |
| 🏁 Goal | Build protein graph representations and train GNNs for structure-based property prediction — the core skill for structural biology ML roles |

### 🆕 From Scratch? Start Here:
1. [CS224W Lecture 1 — Introduction to Graph ML](https://www.youtube.com/watch?v=JAB_plj2rbA) (free, Stanford, ~1h)
2. [PyG Get Started tutorial](https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html) (free, ~2h)
3. Module 00/06 — PyTorch nn.Module (this project)
4. Module 03/01 — Protein structures & coordinates (this project)
5. Module 05/01 — Deep learning patterns (this project)
6. **This notebook**

**Time:** 15–20h | **Difficulty:** ⭐⭐⭐⭐⭐

### 🔗 Cross-References
- **Builds on:** 03/01 PDB parsing and Cα/Cβ coordinate extraction (used directly for graph construction), 05/01 PyTorch training loop and LoRA patterns (GNN training follows the same discipline)
- **Used by:** 07/01 AlphaFold3 architecture — Pairformer and IPA build on graph attention concepts from this notebook; 10/01 OpenFold fine-tuning uses equivariant structure representations studied here
- **Parallel reading:** 07/01 AlphaFold3 (uses GNN concepts heavily in structure module)


## What This Notebook Teaches (Plain English)

**Graphs** = networks of nodes (things) connected by edges (relationships).

A protein is naturally a graph: each amino acid is a node, and two nodes are connected by an edge if they are close in 3D space (< 8 Ångströms apart). This captures the protein's shape without relying on sequence order.

**Graph Neural Networks (GNNs)** = deep learning on graphs. Instead of convolution over pixels (CNN) or tokens (Transformer), GNNs pass messages between connected nodes and update each node's representation based on its neighborhood.

### Why Not Just Use a Sequence Model?

| Sequence model (LSTM/Transformer) | GNN |
|-----------------------------------|-----|
| Sees residues in order: 1,2,3...N | Sees spatial neighborhood: who is physically close |
| Misses long-range 3D contacts | Naturally captures 3D contacts |
| Translation-invariant | Rotation + translation invariant (SE(3)-equivariant) |
| Good for: sequence classification | Good for: structure-based binding prediction |

### Message Passing in Plain English

```
Step 1 (Collect): Each node gathers features from its neighbors
Step 2 (Aggregate): Average (or sum) the neighbor features  
Step 3 (Update): Combine own features + aggregated neighbors → new representation
Step 4 (Repeat): Do this N times to propagate information further
```

After 2 rounds of message passing, each node "knows" about residues within 2 hops (contacts of contacts).

### What You Need Before Starting
- Python + PyTorch basics (from Notebook 00/02 and 05/01)
- Protein structure concepts (from Notebook 03/01)
- Understanding of what a neural network layer does

# Structural Biology ML — GNNs & AlphaFold Concepts
**HackerRank Prep — Theme 6 | Interview-Grade**

Covers: protein graphs, GNN architecture, structure-based property prediction, AlphaFold2 key concepts, embedding proteins for ML.

> **Interview tip:** Even if you can't run AlphaFold2, *explaining* its key innovations (Evoformer, IPA, MSA) demonstrates expert-level knowledge.

---

## 1. Protein as a Graph

> **Why graphs?** Proteins have no natural sequential order in 3D space. Graph representations capture spatial relationships that 1D sequence misses.

### 📖 Reading Guide — Building a Protein Graph from 3D Coordinates

`ca_coords = np.array(ca_coords)`
→ *CA = alpha-carbon. One per residue. We use only CA for the coarse-grained graph (ignores side chains).*

`dist_matrix = np.linalg.norm(ca[:, None] - ca[None, :], axis=-1)`
→ *Pairwise distances between all CA atoms. dist_matrix[i][j] = distance in Å between residue i and j.*

`edges = np.argwhere(dist_matrix < threshold)`
→ *Connect residues that are within 8Å of each other — spatially close residues interact.*

`x[i, aa2i[res]] = 1.0  # one-hot encoding`
→ *One-hot: if residue i is Alanine (index 0), set x[i,0]=1 and all others=0. Converts amino acid letter to a 20-dim vector.*

`edge_index = torch.tensor(edges.T, dtype=torch.long)`
→ *PyTorch Geometric expects edges as shape (2, num_edges): [source_nodes, target_nodes].*

`Data(x=node_features, edge_index=edge_index)`
→ *Package everything into a graph object. x=node features, edge_index=connectivity.*



### 📦 Libraries Used Here

- **`torch_geometric.data.Data`** — A container for a single graph: Data(x=node_features, edge_index=edges, y=label).
- **`torch_geometric.nn.GCNConv`** — Graph Convolutional Network layer — each node aggregates features from its neighbours.
- **`torch_geometric.nn.global_mean_pool`** — Pool all node features into one graph-level vector by taking the mean — needed for graph classification.


In [ ]:
import numpy as np
import torch
from torch_geometric.data import Data

def build_protein_graph(ca_coords, sequence, threshold=8.0):
    """Build protein residue graph from CA coordinates."""
    n = len(ca_coords)
    ca = np.array(ca_coords)
    # Node features: one-hot amino acid (20-dim)
    aa = 'ACDEFGHIKLMNPQRSTVWY'
    aa2i = {a:i for i,a in enumerate(aa)}
    x = np.zeros((n, 20))
    for i, res in enumerate(sequence[:n]):
        if res in aa2i:
            x[i, aa2i[res]] = 1.0
    # Edges: CA-CA distance < threshold
    edges = []
    for i in range(n):
        for j in range(i+1, n):
            d = np.linalg.norm(ca[i] - ca[j])
            if d < threshold:
                edges += [[i,j],[j,i]]
    edge_index = torch.tensor(edges, dtype=torch.long).T if edges else torch.zeros(2,0,dtype=torch.long)
    graph = Data(x=torch.tensor(x, dtype=torch.float),
                 edge_index=edge_index, num_nodes=n)
    return graph

np.random.seed(42)
L = 25
seq = 'ACDEFGHIKLMNPQRSTVWY' + 'ACDEF'
ca = np.cumsum(np.random.randn(L, 3)*3.8, axis=0)
g = build_protein_graph(ca, seq)
print(f"Nodes: {g.num_nodes}, Edges: {g.num_edges}")
print(f"Node features: {g.x.shape}")
print(f"Avg degree: {g.num_edges/g.num_nodes:.1f}")

> **Expected output:** Protein graph statistics: node count, edge count, node feature shape, and average degree (e.g., `Nodes: 154, Edges: 1848, Avg degree: 12.0`)  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

## 2. Graph Neural Network for Protein Properties

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch

class ProteinGNN(nn.Module):
    def __init__(self, in_dim=20, hidden=64, out_dim=1):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.head = nn.Linear(hidden, out_dim)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.bn2 = nn.BatchNorm1d(hidden)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.conv3(x, edge_index))
        return self.head(global_mean_pool(x, batch))

torch.manual_seed(42)
# Batch of 4 protein graphs
graphs = []
for i in range(4):
    n = torch.randint(15,30,(1,)).item()
    x = torch.randn(n, 20)
    ei = torch.randint(0, n, (2, n*3))
    graphs.append(Data(x=x, edge_index=ei))
batch = Batch.from_data_list(graphs)
model = ProteinGNN()
pred = model(batch.x, batch.edge_index, batch.batch)
print(f"Predictions shape: {pred.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. AlphaFold2 Key Concepts — Interview Q&A

In [ ]:
# AlphaFold2 key concepts — interview Q&A code examples
import torch
import torch.nn as nn

# 1. Triangle multiplicative update (simplified)
def triangle_multiplicative(z, mask=None):
    """Simplified triangle update: z_ij = sigmoid(gate) * (sum_k z_ik * z_jk)"""
    B, L, L2, d = z.shape
    # Outgoing: z_ij += sum_k z_ik * z_jk
    update = torch.einsum('bikd,bjkd->bijd', z, z) / L
    return z + update

# 2. FAPE loss (simplified)
def fape_loss(pred_coords, true_coords, eps=1e-8):
    """Frame Aligned Point Error — key AlphaFold2 training loss."""
    diff = pred_coords - true_coords
    dist = (diff**2).sum(-1).sqrt()
    clamped = torch.clamp(dist, max=10.0)
    return clamped.mean()

# 3. lDDT-Cα approximation
def lddt_approx(pred_dists, true_dists, thresholds=[0.5, 1.0, 2.0, 4.0]):
    scores = []
    for t in thresholds:
        preserved = (torch.abs(pred_dists - true_dists) < t).float()
        scores.append(preserved.mean())
    return torch.stack(scores).mean()

torch.manual_seed(42)
L = 20
z = torch.randn(1, L, L, 64)
print("Triangle update output:", triangle_multiplicative(z).shape)

pred = torch.randn(L, 3)
true = pred + torch.randn(L, 3)*0.5
print(f"FAPE loss: {fape_loss(pred, true):.4f}")

pd = (pred.unsqueeze(0) - pred.unsqueeze(1)).norm(dim=-1)
td = (true.unsqueeze(0) - true.unsqueeze(1)).norm(dim=-1)
print(f"lDDT approx: {lddt_approx(pd, td):.4f}")

## 4. Protein Embedding & Similarity Search

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def protein_embedding(sequence, esm_dim=320):
    """Simulate protein embedding via random projection (ESM-2 surrogate)."""
    torch.manual_seed(hash(sequence) % (2**31))
    # In practice: use fair-esm model
    return torch.randn(esm_dim)

def cosine_similarity_search(query_emb, database_embs, top_k=5):
    """Return top-k most similar proteins."""
    q = F.normalize(query_emb.unsqueeze(0), dim=-1)
    db = F.normalize(database_embs, dim=-1)
    sims = (q @ db.T).squeeze(0)
    top_vals, top_idx = sims.topk(top_k)
    return top_idx, top_vals

import pandas as pd
seqs = ['ACDEFGHIKLMNPQRSTVWY'*i for i in range(1, 11)]
embeddings = torch.stack([protein_embedding(s) for s in seqs])
query = protein_embedding('ACDEFGHIKLM')
indices, sims = cosine_similarity_search(query, embeddings)
sim_df = pd.DataFrame({'rank': range(1, len(indices)+1),
                       'db_idx': indices.numpy(),
                       'similarity': sims.detach().numpy()})
print("Similarity search results:")
print(sim_df.to_string(index=False))

## 5. Binding Site Prediction (Graph Classification)

> End-to-end example that ties structure → graph → GNN → prediction.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch
import numpy as np

def make_binding_site_graph(n_residues, has_binding_site=True):
    x = torch.randn(n_residues, 20)
    if has_binding_site:
        # Binding site residues have distinctive features
        site_idx = torch.randint(0, n_residues, (5,))
        x[site_idx, :5] += 3.0
    edge_index = torch.randint(0, n_residues, (2, n_residues*3))
    label = torch.tensor([int(has_binding_site)])
    return Data(x=x, edge_index=edge_index, y=label)

class BindingSiteGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(20, 64)
        self.conv2 = GCNConv(64, 32)
        self.head = nn.Linear(32, 2)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.head(global_mean_pool(x, batch))

torch.manual_seed(42)
dataset = [make_binding_site_graph(np.random.randint(20,40), i%2==0)
           for i in range(50)]
batch = Batch.from_data_list(dataset[:8])
model = BindingSiteGNN()
out = model(batch.x, batch.edge_index, batch.batch)
print(f"Binding site prediction: {out.shape}")
print(f"Predicted classes: {out.argmax(1).tolist()}")

## 5b. GNN Training Loop — Complete Example

> **Interview Q:** *How do you batch graphs of different sizes?*

Unlike images (fixed grid), graphs have variable numbers of nodes and edges. The standard approach is to **concatenate** all graphs in a batch into one large disconnected graph, tracking which nodes belong to which graph with a `batch` index vector.

**Gradient clipping** (`clip_grad_norm_`) is critical for GNNs — message aggregation can cause gradient spikes.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
import numpy as np

# Complete GNN training loop
torch.manual_seed(42)
dataset = []
for i in range(200):
    n = np.random.randint(10, 40)
    x = torch.randn(n, 20)
    if i % 2 == 0:
        x[:, 0] += 2.0  # class signal
    ei = torch.randint(0, n, (2, n*3))
    dataset.append(Data(x=x, edge_index=ei, y=torch.tensor([i%2])))

train_loader = DataLoader(dataset[:160], batch_size=32, shuffle=True)
val_loader = DataLoader(dataset[160:], batch_size=32)

class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(20, 64); self.conv2 = GCNConv(64, 32)
        self.head = nn.Linear(32, 2)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index)); x = F.relu(self.conv2(x, edge_index))
        return self.head(global_mean_pool(x, batch))

model = GNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

for epoch in range(1, 6):
    model.train(); loss_sum = 0; correct = 0
    for b in train_loader:
        opt.zero_grad()
        out = model(b.x, b.edge_index, b.batch)
        loss = F.cross_entropy(out, b.y.squeeze())
        loss.backward(); opt.step()
        loss_sum += loss.item(); correct += (out.argmax(1)==b.y.squeeze()).sum().item()
    model.eval(); val_correct = 0
    with torch.no_grad():
        for b in val_loader:
            out = model(b.x, b.edge_index, b.batch)
            val_correct += (out.argmax(1)==b.y.squeeze()).sum().item()
    print(f"Epoch {epoch}: loss={loss_sum/len(train_loader):.4f} train_acc={correct/160:.3f} val_acc={val_correct/40:.3f}")
    scheduler.step()

## 5c. Protein Graph Visualization

Visualizing graphs is an important debugging step — it confirms your edge construction is correct and shows the structural patterns the GNN must learn.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.utils import to_networkx

# Build a small protein graph for visualization
np.random.seed(42)
L = 15
seq = 'ACDEFGHIKLMNPQR'
ca = np.cumsum(np.random.randn(L, 3) * 3.8, axis=0)

edges = []
edge_weights = []
threshold = 8.0
for i in range(L):
    for j in range(i+1, L):
        d = np.linalg.norm(ca[i]-ca[j])
        if d < threshold:
            edges += [[i,j],[j,i]]
            edge_weights += [d, d]

edge_index = torch.tensor(edges, dtype=torch.long).T
x = torch.eye(L, 20)[:L,:20] if L<=20 else torch.randn(L,20)
graph = Data(x=x, edge_index=edge_index, num_nodes=L)
G = to_networkx(graph, to_undirected=True)

pos = {i: (ca[i,0], ca[i,1]) for i in range(L)}
fig, ax = plt.subplots(figsize=(8,6))
nx.draw(G, pos=pos, ax=ax, with_labels=True, node_color='lightblue',
        node_size=500, labels={i: seq[i] for i in range(L)},
        font_size=8, edge_color='gray', width=0.8)
ax.set_title(f'Protein Graph ({L} residues, threshold={threshold}Å)')
plt.tight_layout()
plt.savefig('/tmp/protein_graph.png', dpi=72)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Average degree: {2*G.number_of_edges()/G.number_of_nodes():.1f}")

## 6. Interview-Grade Reference: Structural Biology ML Stack

In [ ]:
# Structural Biology ML Stack — Reference Implementation
import torch
import torch.nn as nn

print("=" * 60)
print("STRUCTURAL BIOLOGY ML STACK — INTERVIEW REFERENCE")
print("=" * 60)

stack = {
    "Structure Prediction": ["AlphaFold2/3", "ESMFold", "RoseTTAFold"],
    "Graph Neural Networks": ["GCN", "GAT", "GIN", "EGNN", "SE3-Transformer"],
    "Protein Language Models": ["ESM-2", "ProtTrans", "ProteinBERT"],
    "Key Losses": ["FAPE", "lDDT", "RMSD", "TM-score", "DockQ"],
    "Key Operations": ["Triangle attention", "Pairformer", "IPA", "Evoformer"],
}

for category, items in stack.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  - {item}")

print("\n" + "=" * 60)
print("Key interview questions:")
questions = [
    "What is the difference between lDDT and RMSD?",
    "Why does AlphaFold3 use diffusion instead of coordinates directly?",
    "What makes triangle attention O(L^2) instead of O(L^3)?",
    "How does LoRA reduce trainable parameters?",
    "What is equivariance and why does it matter for protein structures?",
]
for i, q in enumerate(questions, 1):
    print(f"  Q{i}: {q}")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- Graph Laplacian theory — L = D - A, eigenvalue spectrum, graph convolution as low-pass filter
- 1-Weisfeiler-Leman (1-WL) isomorphism test — color refinement algorithm, GNN expressivity upper bound
- SE(3) equivariance — Lie groups, irreducible representations (irreps), spherical harmonics in E3NN/MACE
- Over-smoothing — Dirichlet energy, why deep message passing collapses node features, PairNorm fix
- Papers: [MPNN (Gilmer 2017)](https://arxiv.org/abs/1704.01212), [GATv2 (2022)](https://arxiv.org/abs/2105.14491), [EGNN (Satorras 2021)](https://arxiv.org/abs/2102.09844), [AlphaFold2 IPA module](https://www.nature.com/articles/s41586-021-03819-2)

### 2️⃣ Must-Have Popular Resources
- 📘 **Graph Representation Learning** (Hamilton) — [free PDF](https://www.cs.mcgill.ca/~wlh/grl_book/), covers MPNN, GCN, GraphSAGE theory
- 🎓 **CS224W Stanford** (Jure Leskovec) — [YouTube playlist](https://www.youtube.com/playlist?list=PLoROMvodv4rPLKxIpqhjhPgdQy7imNkDn) (free, best GNN course)
- ⭐ **GitHub** [pyg-team/pytorch_geometric](https://github.com/pyg-team/pytorch_geometric) 21k★ — PyG, the standard GNN library
- ⭐ **GitHub** [dmlc/dgl](https://github.com/dmlc/dgl) 13k★ — DGL, backend-agnostic (PyTorch/TensorFlow/MXNet)
- ⭐ **GitHub** [Open-Catalyst-Project/ocp](https://github.com/Open-Catalyst-Project/ocp) 700★ — equivariant GNN benchmarks for materials
- 🤗 **HuggingFace** [graphs-datasets collection](https://huggingface.co/graphs-datasets) — TUDataset, ZINC, OGB in HF format
- 📊 **Kaggle** [Predicting Molecular Properties](https://www.kaggle.com/competitions/champs-scalar-coupling) — J-coupling prediction, GNN baseline

### 3️⃣ Practicals — Hands-On
- 💻 **Exercise**: Implement message passing without PyG — pure NumPy/PyTorch adjacency matrix approach, verify against PyG output
- 💻 **Exercise**: Train GCN on ZINC molecular property prediction — MAE target, compare mean-pooling vs attention-pooling readout
- 💻 **Exercise**: Protein contact map prediction with GNN — residue graph (Cα ≤ 8Å edges), binary edge classification
- 💻 **Exercise**: Implement 1-WL color refinement — count distinguishable graphs in a set, show GNN limitation on symmetric graphs
- 🔗 **GitHub** [OGB leaderboards](https://ogb.stanford.edu/docs/leader_graphprop/) — ogbg-molhiv, ogbg-molpcba benchmarks
- 📊 **Kaggle** [NeurIPS 2022 Open Catalyst](https://www.kaggle.com/competitions/open-catalyst-challenge) — equivariant GNN for energy prediction
- 🤗 **HuggingFace Space** [Molecular GNN demo](https://huggingface.co/spaces/RuibresMolecularAI/molecular-property-prediction)

### 4️⃣ Real-World Problems
- 🏭 **Industry**: computational biology ML teams — graph + diffusion model for drug-protein interaction (AlphaFold3 successor)
- 🏭 **Industry**: Schrödinger FEP+ — force field GNNs for alchemical free energy perturbation
- 🏭 **Industry**: D.E. Shaw Research — equivariant GNNs for MD force field learning (Anton-3 simulations)
- 📊 **Dataset**: [ZINC15](https://zinc15.docking.org/) — 1.5B purchasable compounds, GNN molecular property benchmarks
- 📊 **Dataset**: [PDBbind](http://www.pdbbind.org.cn/) — 23k protein-ligand complexes with measured Kd values
- 🔬 **Application**: Virtual screening — graph-level classification (active/inactive), enrichment factor, ROC-AUC vs BEDROC

### 5️⃣ Interview Tips
- ❓ **Must know**: Why message passing ≤ 1-WL — two non-isomorphic graphs with same degree sequence are indistinguishable without structural features
- ❓ **Must know**: Over-smoothing depth — empirically starts ~7-10 layers; Dirichlet energy of node features → 0 with depth
- ❓ **Must know**: EGNN vs GVP-GNN for structure — EGNN uses equivariant coordinates directly; GVP uses scalar+vector features; both are SE(3)-equivariant
- ⚠️ **Common mistake**: Using `add` pooling instead of `mean` for graph-level tasks — `add` is sensitive to graph size variation
- ⚠️ **Common mistake**: Building protein graphs with only backbone atoms — include Cβ atoms and sidechain contacts for better ML signal
- 💡 **Pro tip**: Always mention 1-WL limitation in GNN interviews — interviewers reward this; follow up with higher-order WL or structural features (RWSE, LSPE)
- 💡 **Pro tip**: For protein structure graphs, 8Å Cβ–Cβ threshold with RBF-encoded distances as edge features is the standard baseline

### 6️⃣ Hot Industry Topics
- 🔥 **Trending** [atomicarchitects/equiformer_v2](https://github.com/atomicarchitects/equiformer_v2) — state-of-art equivariant GNN for OC20/OC22 datasets, EquiformerV2
- 🔥 **Trending** [microsoft/unimol](https://github.com/dptech-corp/Uni-Mol) 1k★ — MSRA Uni-Mol transformer for molecules + proteins, SE(3)-invariant
- 🔥 **Trending** [microsoft/graphormer](https://github.com/microsoft/Graphormer) 2k★ — attention with graph structural encodings, won KDD Cup 2021
- 🔥 **Trending** [RosettaFold-All-Atom / FrameDiff](https://github.com/jasonkyuyim/se3_diffusion) — SE(3) diffusion for protein backbone generation
- 🚀 **Build this**: Protein binding site predictor — build residue graph from PDB (Cα nodes, Cβ–Cβ ≤ 8Å edges), train 3-layer GATv2 with RBF edge features, ROC-AUC on sc-PDB binding sites
- 🚀 **Build this**: Molecular property GNN — load ZINC from PyG, GCN + global mean pooling, predict logP, compare to fingerprint+RF baseline, interpret with GNNExplainer


In [ ]:
# Resources cell — key references for Structural ML + GNNs
print("KEY RESOURCES — Module 06: Structural ML + GNNs")
print()
resources = {
    "Papers": [
        "Jumper et al. 2021 — AlphaFold2 (Nature)",
        "Ingraham et al. 2019 — Generative models for graph-based protein design",
        "Satorras et al. 2021 — E(n) Equivariant GNNs",
        "Xu et al. 2019 — How Powerful are GNNs? (GIN)",
        "Veličković et al. 2018 — Graph Attention Networks",
    ],
    "Code": [
        "PyTorch Geometric: pytorch-geometric.readthedocs.io",
        "ESM: github.com/facebookresearch/esm",
        "AlphaFold: github.com/deepmind/alphafold",
    ],
    "Datasets": [
        "PDB: rcsb.org",
        "SAbDab: antibody structures",
        "ProteinNet: sequence + structure benchmark",
        "CATH/SCOP: protein structure classification",
    ]
}
for cat, items in resources.items():
    print(f"\n{cat}:")
    for item in items:
        print(f"  * {item}")

# Edges, threshold, nProtein for the fill system reference
edges = 'CA-CA edges within threshold distance'
threshold = 8.0  # Angstrom
nProtein = 1000  # typical PDB dataset size
print(f"\nGraph construction: {edges}, threshold={threshold}Å")
print(f"Typical dataset: {nProtein} protein structures")

# 🌍 Real World Problems — Structural ML & Graph Neural Networks

---

## Problem 1 — Drug-Target Interaction Prediction (DTI)
**Dataset**: [BindingDB](https://www.bindingdb.org/) | [Kaggle: Drug-Target Interaction](https://www.kaggle.com/datasets/dhirajmandal10/drug-target-interaction-dataset)
**GitHub**: [github.com/DeepPurpose/DeepPurpose](https://github.com/DeepPurpose/DeepPurpose)
**Paper**: Chemical Reviews 2025 — GNNs in Drug Discovery
**Skills**: Protein graphs, message passing, molecular fingerprints

```python
# Drug-target interaction: given a protein structure + drug molecule,
# predict binding affinity (regression) or binding/non-binding (classification)

# Using the approach from this notebook's GNN + DeepPurpose-style workflow
import numpy as np
import torch

# Protein side: use ProteinGNN from this notebook
# Drug side: represent as a molecular graph (atoms=nodes, bonds=edges)
# Combined: concatenate protein + drug embeddings → MLP → affinity score

# Simplified drug fingerprint representation (Morgan fingerprints)
def morgan_fingerprint(smiles: str, radius: int = 2, bits: int = 2048) -> np.ndarray:
    """
    Compute Morgan (circular) fingerprint from SMILES string.
    Requires rdkit: pip install rdkit
    """
    try:
        from rdkit import Chem
        from rdkit.Chem import AllChem
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(bits)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=bits)
        return np.array(fp)
    except ImportError:
        # Fallback: random fingerprint for demo
        np.random.seed(hash(smiles) % 2**31)
        return np.random.randint(0, 2, bits).astype(np.float32)

# Example drugs (SMILES) and targets
drugs = {
    'Erlotinib':   'c1ccc2c(c1)cccc2Nc1ncnc2cc(OCCO)c(OCCO)cc12',
    'Imatinib':    'Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1',
    'Gefitinib':   'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',
}
proteins = {
    'EGFR':  'MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNK',  # fragment
    'BCR-ABL': 'MAAVILESIFLKRSQQKKKTSPLNFKKRLFLLTVHKLSYYEYDFER',
}

print("DTI Prediction Demo:")
for drug_name, smiles in drugs.items():
    fp = morgan_fingerprint(smiles)
    print(f"  {drug_name}: fingerprint dim={len(fp)}, active bits={fp.sum():.0f}")

# YOUR TASK: complete DTI model
# 1. Protein embedding: use ProteinGNN(from_coords()) from this notebook
# 2. Drug embedding: Morgan fingerprint (above) → MLP
# 3. Combined: concat embeddings → 2-layer MLP → binding score
# 4. Train on BindingDB IC50 data
```

---

## Problem 2 — Protein Function Prediction (GO Annotation)
**Dataset**: [Kaggle: CAFA 5 Protein Function Prediction](https://www.kaggle.com/competitions/cafa-5-protein-function-prediction) — $57K prize
**Hugging Face**: [facebook/esm2_t33_650M_UR50D](https://huggingface.co/facebook/esm2_t33_650M_UR50D)
**GitHub**: [github.com/snap-stanford/GIANT](https://github.com/snap-stanford/GIANT)
**Skills**: Multi-label classification, GNN + sequence model fusion

```python
# CAFA 5: predict Gene Ontology (GO) terms for proteins
# This is a multi-label classification (one protein → many GO terms)
# Top teams in 2024 used ESM2 embeddings + GNN on protein interaction networks

# Gene Ontology terms (simplified to 3 categories)
GO_terms = {
    'MF': 'Molecular Function',  # What the protein does chemically
    'BP': 'Biological Process',  # What pathway/process it's involved in
    'CC': 'Cellular Component',  # Where in the cell it operates
}

# Approach: ESM2 embeddings + protein-protein interaction (PPI) network GNN
# Step 1: Get ESM2 embeddings for each protein sequence
# from transformers import AutoTokenizer, AutoModel
# tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
# model = AutoModel.from_pretrained("facebook/esm2_t33_650M_UR50D")

# Step 2: Build PPI network as a graph (from STRING database)
# Each protein = node, interactions = edges
# Run GNN (from this notebook) on the PPI graph

# Step 3: Combine ESM2 embedding + GNN output → multi-label classifier

# Simplified demo:
import numpy as np
np.random.seed(42)
n_proteins = 100
n_go_terms = 50
esm2_dim = 1280  # ESM2-650M embedding dimension

# Simulate ESM2 embeddings (real: run ESM2 on each sequence)
protein_embeddings = np.random.randn(n_proteins, esm2_dim)

# Multi-label targets (each protein can have multiple GO terms)
y_multilabel = (np.random.rand(n_proteins, n_go_terms) > 0.8).astype(int)

from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(protein_embeddings, y_multilabel,
                                            test_size=0.2, random_state=42)
clf = MultiOutputClassifier(LogisticRegression(max_iter=200))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
print(f"Micro F1: {f1_score(y_te, y_pred, average='micro'):.4f}")
print(f"Macro F1: {f1_score(y_te, y_pred, average='macro'):.4f}")
print("\nKaggle CAFA 5: kaggle.com/competitions/cafa-5-protein-function-prediction")
```

---

## Problem 3 — Molecular Property Prediction (QM9 Dataset)
**Dataset**: [QM9 dataset](https://figshare.com/collections/Quantum_chemistry_structures_and_properties_of_134_kilo_molecules/978904) — 134K molecules, 12 quantum properties
**GitHub**: [github.com/pyg-team/pytorch_geometric](https://github.com/pyg-team/pytorch_geometric) (PyG)
**Skills**: Molecular graphs, GNN regression, transfer learning

```python
# QM9: predict molecular properties (dipole moment, atomization energy, etc.)
# from molecular graphs (atoms = nodes, bonds = edges)
# This is the standard GNN benchmark dataset

# Using PyTorch Geometric:
# pip install torch_geometric
# from torch_geometric.datasets import QM9
# from torch_geometric.loader import DataLoader
# dataset = QM9(root='data/QM9')
# print(f"Number of molecules: {len(dataset)}")
# print(f"Node features: {dataset[0].x.shape}")      # (n_atoms, 11)
# print(f"Edge features: {dataset[0].edge_attr.shape}")  # (n_bonds*2, 4)
# print(f"Targets: {dataset[0].y.shape}")             # (1, 19) — 12+7 properties

# YOUR TASK: build a GNN (similar to ProteinGNN in this notebook) for QM9
# Use the MessagePassingLayer from this notebook
# Adapt it for molecular graphs (atom features → bond features → molecular property)

# Atom feature mapping (11 features per atom):
atom_features = {
    'atom_type':     [0, 1, 2, 3, 4],    # H, C, N, O, F
    'atomic_number': 'continuous',
    'acceptor':      'binary',
    'donor':         'binary',
    'aromatic':      'binary',
    'hybridization': [0, 1, 2],           # sp, sp2, sp3
    'num_Hs':        'continuous',
}
print("QM9 is the standard benchmark for molecular property GNNs")
print("GitHub PyG: github.com/pyg-team/pytorch_geometric")
print("Load with: QM9(root='data/QM9') from torch_geometric.datasets")
```

---

## Resources
| Resource | URL | Purpose |
|----------|-----|---------|
| Kaggle: CAFA 5 | kaggle.com/competitions/cafa-5-protein-function-prediction | $57K GO annotation |
| Kaggle: DTI | kaggle.com/datasets/dhirajmandal10/drug-target-interaction-dataset | Drug-target pairs |
| GitHub: DeepPurpose | github.com/DeepPurpose/DeepPurpose | DTI toolkit |
| GitHub: PyG | github.com/pyg-team/pytorch_geometric | GNN framework |
| HF: ESM2 650M | huggingface.co/facebook/esm2_t33_650M_UR50D | Best protein embeddings |
| HF: ESM2 8M | huggingface.co/facebook/esm2_t6_8M_UR50D | Lightweight (runs on CPU) |
| BindingDB | bindingdb.org | Drug-target affinities |
| STRING DB | string-db.org | Protein-protein interactions |
| AlphaFold DB | alphafold.ebi.ac.uk | Predicted structures (200M+) |


## 🎤 Interview Questions

**Q1 (Easy):** What is the advantage of representing a protein as a graph rather than a sequence?
<details><summary>Answer</summary>Graphs capture 3D spatial relationships (which residues are physically close) that sequences miss. Two residues far apart in sequence may be adjacent in 3D space and interact directly.</details>

**Q2 (Easy):** What are the three steps of message passing in a GNN?
<details><summary>Answer</summary>1) Message: compute messages from neighbors. 2) Aggregate: combine messages (sum/mean/max). 3) Update: update node features using aggregated messages + current features.</details>

**Q3 (Medium):** Why is SE(3)-equivariance important for protein structure prediction?
<details><summary>Answer</summary>Rotating a protein in space shouldn't change the predicted properties (invariance for scalars like energy) or should rotate predictions the same way (equivariance for vectors like forces). Without this, the model must learn all rotations from data.</details>

**Q4 (Medium):** How would you choose the graph construction method (k-NN vs distance threshold) for a protein GNN?
<details><summary>Answer</summary>k-NN gives each node the same number of edges (good for batching), while distance threshold better reflects physical contacts. k-NN with k=10-30 and distance threshold of 8-10Å on Cα atoms are both common choices.</details>

**Q5 (Hard):** Design a GNN that predicts per-residue B-factors from protein structure. What architecture choices matter most?
<details><summary>Answer</summary>Node features: amino acid type + secondary structure + relative solvent accessibility. Edge features: distance + angle. Architecture: 4-6 MPNN layers with skip connections to avoid over-smoothing. Key choice: include side-chain atoms as separate nodes or use Cα-only graph with richer node features. B-factors correlate with local flexibility, so the model needs to capture local packing density.</details>

## Mastery Check

Before moving on, make sure you can:
1. explain why proteins can be represented as graphs
2. distinguish invariance from equivariance
3. describe what messages are passed along edges
4. justify one design choice in the graph construction


## Notebook Complete

**You can now:**
- Represent proteins as graphs and build protein contact graphs from PDB coordinates
- Train a GCN to predict residue-level or graph-level properties

**Where this fits:**
- Previous: 05_deep_learning_finetuning
- Next: 02_gnn_deep_dive — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes